In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import xarray as xr


MANIFEST = Path("../data/temperature/temperature_input_manifest_35cities.csv")

OUT_BASE = Path(
    "../outputs/heatwave"
)

DATES = pd.date_range("2016-01-01", "2025-12-31")
YEARS = list(range(2016, 2026))
DOY = DATES.dayofyear.to_numpy()


DOY_PERIOD = 366
WINDOW = 15
MIN_DURATION = 3


MIN_BASELINE_COVERAGE = 0.90

FIXED_TX = 35.0
FIXED_TN = 25.0

EXPECTED_CITIES = 35
EXPECTED_POINTS = 652

SCHEMES = [
    "local_p90",
    "local_p95",
    "city_p90",
    "city_p95",
    "fixed_tx35_tn25",
]

TYPES = ["day", "night", "compound"]

METRICS = [
    "HW_days",
    "HW_events",
    "HW_max_duration",
    "HW_mean_intensity",
    "HW_max_intensity",
    "HWDD",
]

EXTRA = [
    "HW_events_touching",
    "HW_max_full_event_duration",
]

KEY = [
    "city", "cell_id", "year", "scheme", "heatwave_type"
]

EVENT_COLUMNS = [
    "city", "cell_id", "lat", "lon",
    "scheme", "threshold_pct", "heatwave_type",
    "event_id", "start_date", "end_date",
    "start_year", "end_year", "duration_days",
    "cross_year",
    "record_start_censored",
    "record_end_censored",
    "all_touched_years_complete",
    "mean_intensity_degC",
    "max_intensity_degC",
    "HWDD_degC_day",
]


def require(ok, message):
    if not ok:
        raise RuntimeError(message)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def runs(mask):
    x = np.asarray(mask, dtype=bool)
    if not x.size:
        return []

    starts = np.flatnonzero(x & ~np.r_[False, x[:-1]])
    ends = np.flatnonzero(x & ~np.r_[x[1:], False]) + 1

    return list(zip(starts, ends))


def load_city(path, city):
    with xr.open_dataset(path) as ds:
        a = ds.load()

    require(
        a.attrs.get("city") == city,
        f"City mismatch: {path}",
    )
    require(
        pd.DatetimeIndex(a.time.values).equals(DATES),
        f"Time axis incomplete or duplicated: {city}",
    )
    require(
        pd.Index(a.cell_id.values).is_unique,
        f"Duplicate cell_id: {city}",
    )
    require(
        a.lat.dims == a.lon.dims == ("cell_id",),
        "Wrong dimensions of study-point coordinates.",
    )
    require(
        np.isfinite(a.lat.values).all()
        and np.isfinite(a.lon.values).all(),
        "Study-point coordinates missing.",
    )

    for temp, status in [
        ("tx_C", "tx_status"),
        ("tn_C", "tn_status"),
    ]:
        require(
            a[temp].dims == a[status].dims == ("time", "cell_id"),
            "Wrong temperature dimensions.",
        )
        require(
            a[temp].attrs.get("units") == "degree_Celsius",
            "Temperature must be in degrees Celsius.",
        )

        v = a[temp].values
        require(not np.isinf(v).any(), "Infinite temperature values present.")

        finite_values = v[np.isfinite(v)]
        require(
            ((finite_values > -100) & (finite_values < 70)).all(),
            "Abnormal temperature values.",
        )
        require(
            np.isin(a[status].values, np.arange(7)).all(),
            "Unknown temperature status code.",
        )

        np.testing.assert_array_equal(
            np.isfinite(v),
            a[status].values < 2,
        )

    valid = (
        np.isfinite(a.tx_C.values)
        & np.isfinite(a.tn_C.values)
    )

    np.testing.assert_array_equal(
        a.pair_valid.values,
        valid.astype("uint8"),
    )

    require(
        not (valid & (a.tx_C.values < a.tn_C.values)).any(),
        "TX<TN found; stopping.",
    )

    return a


def write_verified(ds, path):
    for name in ds.variables:
        ds[name].encoding = {}

    encoding = {
        name: dict(
            zlib=True,
            complevel=4,
            _FillValue=np.nan if ds[name].dtype.kind == "f" else None,
        )
        for name in ds.data_vars
    }

    pending = path.with_name(path.stem + ".pending.nc")

    ds.to_netcdf(
        pending,
        engine="netcdf4",
        encoding=encoding,
    )

    with xr.open_dataset(pending) as check:
        xr.testing.assert_identical(ds, check.load())

    pending.replace(path)


def save_csv(df, path):
    pending = path.with_name(path.stem + ".pending.csv")

    df.to_csv(
        pending,
        index=False,
        encoding="utf-8-sig",
        float_format="%.17g",
    )

    check = pd.read_csv(
        pending,
        float_precision="round_trip",
    )

    pd.testing.assert_frame_equal(
        df.reset_index(drop=True),
        check,
        check_dtype=False,
        check_exact=True,
    )

    pending.replace(path)


def make_thresholds(tx, tn):
    n = tx.shape[1]

    windows = []
    for d in range(1, DOY_PERIOD + 1):
        delta = np.abs(DOY - d)
        circular_distance = np.minimum(delta, DOY_PERIOD - delta)

        windows.append(
            np.flatnonzero(circular_distance <= WINDOW)
        )

    expected = np.array(
        [len(ii) for ii in windows],
        dtype="int32",
    )

    nx = np.stack([
        np.isfinite(tx[ii]).sum(axis=0)
        for ii in windows
    ]).astype("int32")

    nn = np.stack([
        np.isfinite(tn[ii]).sum(axis=0)
        for ii in windows
    ]).astype("int32")

    annual_pair = np.stack([
        (
            np.isfinite(tx[DATES.year == y])
            & np.isfinite(tn[DATES.year == y])
        ).mean(axis=0)
        for y in YEARS
    ])

    min_year = annual_pair.min(axis=0)

    eligible = (
        (min_year >= MIN_BASELINE_COVERAGE)
        & (
            nx >= np.ceil(
                expected[:, None] * MIN_BASELINE_COVERAGE
            )
        ).all(axis=0)
        & (
            nn >= np.ceil(
                expected[:, None] * MIN_BASELINE_COVERAGE
            )
        ).all(axis=0)
    )

    require(
        eligible.any(),
        "No fixed reference grid cell in this city meets the completeness criteria.",
    )


    qx = np.full(
        (len(SCHEMES), DOY_PERIOD, n),
        np.nan,
        dtype="float32",
    )
    qn = np.full_like(qx, np.nan)

    cx = np.broadcast_to(nx, qx.shape).copy()
    cn = np.broadcast_to(nn, qn.shape).copy()

    ids = np.flatnonzero(eligible)

    for d, ii in enumerate(windows):
        x = tx[np.ix_(ii, ids)]
        z = tn[np.ix_(ii, ids)]

        for k, p in [(0, .90), (1, .95)]:
            qx[k, d, ids] = np.nanquantile(
                x, p, axis=0, method="linear"
            )
            qn[k, d, ids] = np.nanquantile(
                z, p, axis=0, method="linear"
            )


        for k, p in [(2, .90), (3, .95)]:
            qx[k, d, ids] = np.nanquantile(
                x, p, method="linear"
            )
            qn[k, d, ids] = np.nanquantile(
                z, p, method="linear"
            )
            cx[k, d, :] = int(np.isfinite(x).sum())
            cn[k, d, :] = int(np.isfinite(z).sum())

    qx[4, :, ids] = FIXED_TX
    qn[4, :, ids] = FIXED_TN

    cx[4, :, :] = -1
    cn[4, :, :] = -1

    return qx, qn, cx, cn, expected, eligible, min_year


def build_threshold_dataset(a, city, input_hash):
    tx = a.tx_C.values.astype(float)
    tn = a.tn_C.values.astype(float)

    (
        qx, qn, cx, cn,
        expected, eligible, min_year,
    ) = make_thresholds(tx, tn)

    coords = dict(
        scheme=SCHEMES,
        doy=np.arange(1, DOY_PERIOD + 1),
        cell_id=a.cell_id.values,
        lat=("cell_id", a.lat.values),
        lon=("cell_id", a.lon.values),
    )

    attrs = dict(
        city=city,
        baseline="2016-2025",
        window_plus_minus_days=WINDOW,
        min_duration_days=MIN_DURATION,
        input_sha256=input_hash,
        threshold_operator=">=",
        minimum_baseline_coverage=MIN_BASELINE_COVERAGE,
        quantile_method="linear",
    )

    threshold_ds = xr.Dataset(
        {
            "threshold_tx_C": (
                ("scheme", "doy", "cell_id"), qx
            ),
            "threshold_tn_C": (
                ("scheme", "doy", "cell_id"), qn
            ),
            "threshold_sample_count_tx": (
                ("scheme", "doy", "cell_id"), cx
            ),
            "threshold_sample_count_tn": (
                ("scheme", "doy", "cell_id"), cn
            ),
            "expected_window_days_per_cell": (
                "doy", expected
            ),
            "baseline_eligible": (
                "cell_id", eligible.astype("uint8")
            ),
            "minimum_year_pair_coverage": (
                "cell_id", min_year
            ),
        },
        coords=coords,
        attrs=attrs,
    )

    for name in ["threshold_tx_C", "threshold_tn_C"]:
        threshold_ds[name].attrs["units"] = "degree_Celsius"

    reference = pd.DataFrame(dict(
        city=city,
        cell_id=a.cell_id.values,
        lat=a.lat.values,
        lon=a.lon.values,
        baseline_eligible=eligible,
        city_reference_cell=eligible,
        minimum_year_pair_coverage=min_year,
    ))

    return threshold_ds, reference


def classify(tx, tn, qx, qn):
    valid = (
        np.isfinite(tx)
        & np.isfinite(tn)
        & np.isfinite(qx)
        & np.isfinite(qn)
    )

    hx = valid & (tx >= qx)
    hn = valid & (tn >= qn)

    candidates = {
        "day": hx & ~hn,
        "night": hn & ~hx,
        "compound": hx & hn,
    }

    result = {}

    for kind, candidate in candidates.items():
        flag = np.zeros(len(tx), dtype="float32")

        for s, e in runs(candidate):
            if e - s >= MIN_DURATION:
                flag[s:e] = 1

        uncertain = ~valid

        for s, e in runs(candidate | ~valid):
            if (~valid[s:e]).any():
                uncertain[s:e] = True

        flag[uncertain] = np.nan
        result[kind] = flag

    return result, valid


def process_city(a, th, city):
    tx = a.tx_C.values.astype(float)
    tn = a.tn_C.values.astype(float)

    qx = th.threshold_tx_C.values
    qn = th.threshold_tn_C.values
    eligible = th.baseline_eligible.values.astype(bool)

    shape = (
        len(SCHEMES),
        len(DATES),
        len(a.cell_id),
    )

    flags = {
        kind: np.full(shape, np.nan, dtype="float32")
        for kind in TYPES
    }

    valid_all = np.zeros(shape, dtype="uint8")
    annual, events = [], []

    year_indices = {
        y: np.flatnonzero(DATES.year == y)
        for y in YEARS
    }

    for k, scheme in enumerate(SCHEMES):
        pct = (
            np.nan if scheme.startswith("fixed_")
            else 95 if scheme.endswith("p95")
            else 90
        )

        for j, cell_id in enumerate(a.cell_id.values):
            x = tx[:, j]
            z = tn[:, j]

            thx = qx[k, DOY - 1, j]
            thn = qn[k, DOY - 1, j]

            f, valid = classify(x, z, thx, thn)
            valid_all[k, :, j] = valid.astype("uint8")

            known = np.logical_and.reduce([
                np.isfinite(f[t])
                for t in TYPES
            ])

            year_complete = {
                y: bool(valid[ii].all() and known[ii].all())
                for y, ii in year_indices.items()
            }

            for kind in TYPES:
                flag = f[kind]
                flags[kind][k, :, j] = flag
                rr = runs(flag == 1)

                require(
                    all(e - s >= MIN_DURATION for s, e in rr),
                    "Continuous event shorter than 3 days.",
                )

                if kind == "day":
                    excess = x - thx
                elif kind == "night":
                    excess = z - thn
                else:
                    excess = (x - thx) + (z - thn)

                for eid, (s, e) in enumerate(rr, 1):
                    ev = excess[s:e]
                    sy = int(DATES[s].year)
                    ey = int(DATES[e - 1].year)

                    require(
                        np.isfinite(ev).all() and (ev >= 0).all(),
                        "Abnormal event intensity.",
                    )

                    events.append(dict(
                        city=city,
                        cell_id=int(cell_id),
                        lat=float(a.lat.values[j]),
                        lon=float(a.lon.values[j]),
                        scheme=scheme,
                        threshold_pct=pct,
                        heatwave_type=kind,
                        event_id=eid,
                        start_date=DATES[s].strftime("%Y-%m-%d"),
                        end_date=DATES[e - 1].strftime("%Y-%m-%d"),
                        start_year=sy,
                        end_year=ey,
                        duration_days=int(e - s),
                        cross_year=sy != ey,
                        record_start_censored=s == 0,
                        record_end_censored=e == len(DATES),
                        all_touched_years_complete=all(
                            year_complete[y]
                            for y in range(sy, ey + 1)
                        ),
                        mean_intensity_degC=float(ev.mean()),
                        max_intensity_degC=float(ev.max()),
                        HWDD_degC_day=float(ev.sum()),
                    ))

                for y, ii in year_indices.items():
                    lo = int(ii[0])
                    hi = int(ii[-1]) + 1

                    touching = [
                        (s, e) for s, e in rr
                        if s < hi and e > lo
                    ]

                    portions = [
                        min(e, hi) - max(s, lo)
                        for s, e in touching
                    ]

                    v = excess[ii][flag[ii] == 1]
                    complete = year_complete[y]

                    if complete:
                        metrics = dict(
                            HW_days=int((flag[ii] == 1).sum()),

                            HW_events=sum(
                                lo <= s < hi for s, e in rr
                            ),

                            HW_events_touching=len(touching),

                            HW_max_duration=max(
                                portions, default=0
                            ),

                            HW_max_full_event_duration=max(
                                [e - s for s, e in touching],
                                default=0,
                            ),

                            HW_mean_intensity=(
                                float(v.mean()) if len(v) else 0.
                            ),
                            HW_max_intensity=(
                                float(v.max()) if len(v) else 0.
                            ),
                            HWDD=(
                                float(v.sum()) if len(v) else 0.
                            ),
                        )
                    else:
                        metrics = {
                            m: np.nan
                            for m in METRICS + EXTRA
                        }

                    status = (
                        "baseline_insufficient"
                        if not eligible[j]
                        else "complete"
                        if complete
                        else "incomplete_year"
                        if not valid[ii].all()
                        else "boundary_uncertainty"
                    )

                    annual.append(dict(
                        city=city,
                        cell_id=int(cell_id),
                        lat=float(a.lat.values[j]),
                        lon=float(a.lon.values[j]),
                        year=y,
                        scheme=scheme,
                        threshold_pct=pct,
                        heatwave_type=kind,
                        baseline_eligible=bool(eligible[j]),
                        expected_days=len(ii),
                        valid_days=int(valid[ii].sum()),
                        missing_or_invalid_days=int(
                            (~valid[ii]).sum()
                        ),
                        unknown_classification_days=int(
                            np.isnan(flag[ii]).sum()
                        ),
                        annual_complete=complete,
                        quality_status=status,
                        **metrics,
                    ))

                require(
                    sum(
                        int((flag[ii] == 1).sum())
                        for ii in year_indices.values()
                    )
                    == sum(e - s for s, e in rr),
                    "Cross-year day allocation is not conserved.",
                )

    require(
        (
            sum((flags[t] == 1).astype(int) for t in TYPES)
            <= 1
        ).all(),
        "The three event types overlap.",
    )

    attrs = dict(th.attrs)
    attrs.update(
    )

    daily = xr.Dataset(
        {
            f"heatwave_{t}": (
                ("scheme", "time", "cell_id"), flags[t]
            )
            for t in TYPES
        },
        coords=dict(
            scheme=SCHEMES,
            time=DATES,
            cell_id=a.cell_id.values,
            lat=("cell_id", a.lat.values),
            lon=("cell_id", a.lon.values),
        ),
        attrs=attrs,
    )

    daily["valid_detection_input"] = (
        ("scheme", "time", "cell_id"), valid_all
    )

    for t in TYPES:
        daily[f"heatwave_{t}"].attrs["meaning"] = "1 = heatwave day, 0 = not, NaN = unknown"

    return (
        pd.DataFrame(annual),
        pd.DataFrame(events, columns=EVENT_COLUMNS),
        daily,
    )


def run_all():
    require(
        MANIFEST.is_file(),
        f"Temperature manifest not found: {MANIFEST}",
    )

    manifest_hash = sha256(MANIFEST)
    m = pd.read_csv(MANIFEST)

    require(
        {"city", "output_nc", "output_sha256"}.issubset(m.columns),
        "Input manifest lacks required columns.",
    )

    require(
        m[["city", "output_nc", "output_sha256"]]
        .notna().all().all(),
        "Input manifest contains null values.",
    )

    require(
        len(m) == EXPECTED_CITIES and m.city.is_unique,
        "Abnormal number of cities in the manifest, or cities not unique.",
    )

    require(
        not m.output_nc.duplicated().any(),
        "Multiple cities point to the same temperature file.",
    )

    m["output_nc"] = [
        str((MANIFEST.parent / p).resolve())
        for p in m.output_nc
    ]

    for r in m.itertuples(index=False):
        require(
            Path(r.output_nc).is_file(),
            f"Input does not exist: {r.output_nc}",
        )
        require(
            sha256(r.output_nc) == r.output_sha256,
            f"Temperature file hash mismatch: {r.city}",
        )

    out = OUT_BASE
    (out / "cities").mkdir(
        parents=True,
        exist_ok=True,
    )

    m.to_csv(
        out / "input_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

    summary = dict(
        status="running",
        input_manifest=MANIFEST.as_posix(),
        input_manifest_sha256=manifest_hash,
        schemes=SCHEMES,
        baseline_years=YEARS,
        min_duration=MIN_DURATION,
        window_plus_minus_days=WINDOW,
        threshold_operator=">=",
        baseline_coverage_rule=MIN_BASELINE_COVERAGE,
        fixed_tx_C=FIXED_TX,
        fixed_tn_C=FIXED_TN,
    )

    def save():
        (out / "summary.json").write_text(
            json.dumps(
                summary,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    save()
    print("Output folder:", out, flush=True)

    tables, event_tables = [], []
    references, city_rows = [], []

    try:
        for i, r in enumerate(m.itertuples(index=False), 1):
            print(
                f"[{i}/{len(m)}] {r.city}: "
                "thresholds, events and annual statistics...",
                flush=True,
            )

            a = load_city(Path(r.output_nc), r.city)

            folder = out / "cities" / r.city
            folder.mkdir(exist_ok=True)

            th, ref = build_threshold_dataset(
                a, r.city, r.output_sha256
            )

            df, ef, daily = process_city(
                a, th, r.city
            )

            require(
                len(df)
                == len(a.cell_id) * len(YEARS)
                * len(SCHEMES) * len(TYPES),
                "Abnormal number of city-year rows.",
            )

            require(
                not df.duplicated(KEY).any(),
                "Duplicate city-year keys.",
            )

            require(
                df.loc[
                    ~df.annual_complete, METRICS + EXTRA
                ].isna().all().all(),
                "Incomplete years were written as valid metrics.",
            )

            require(
                np.isfinite(
                    df.loc[
                        df.annual_complete, METRICS + EXTRA
                    ].to_numpy(float)
                ).all(),
                "Metrics missing for complete years.",
            )

            write_verified(
                th,
                folder / "thresholds.nc",
            )
            write_verified(
                daily,
                folder / "daily_classification.nc",
            )

            save_csv(
                ref,
                folder / "reference_cells.csv",
            )
            save_csv(
                df,
                folder / "annual_metrics_long.csv",
            )
            save_csv(
                ef,
                folder / "events_continuous.csv",
            )
            save_csv(
                ef.loc[ef.cross_year.eq(True)],
                folder / "cross_year_events.csv",
            )

            require(
                sha256(r.output_nc) == r.output_sha256,
                f"Input changed during the run: {r.city}",
            )

            tables.append(df)
            event_tables.append(ef)
            references.append(ref)

            info = dict(
                city=r.city,
                points=len(a.cell_id),
                baseline_eligible=int(
                    ref.baseline_eligible.sum()
                ),
                annual_rows=len(df),
                confirmed_events=len(ef),
                cross_year_events=int(
                    ef.cross_year.eq(True).sum()
                ),
                incomplete_grid_years=len(
                    df.loc[
                        ~df.annual_complete,
                        ["cell_id", "year"],
                    ].drop_duplicates()
                ),
            )

            city_rows.append(info)

            save_csv(
                pd.DataFrame(city_rows),
                out / "city_progress.csv",
            )

            summary["cities_finished"] = i
            save()

            print(
                f"  Done: reference grid cells "
                f"{info['baseline_eligible']}/{info['points']}; "
                f"cross-year events {info['cross_year_events']}.",
                flush=True,
            )

        result = pd.concat(tables, ignore_index=True)
        ref = pd.concat(references, ignore_index=True)
        ev = pd.concat(event_tables, ignore_index=True)

        require(
            len(ref) == EXPECTED_POINTS
            and not ref.duplicated(["city", "cell_id"]).any(),
            "Abnormal total number of study points or keys.",
        )

        require(
            len(result)
            == EXPECTED_POINTS * len(YEARS)
            * len(SCHEMES) * len(TYPES)
            and not result.duplicated(KEY).any(),
            "Abnormal number of rows or keys in the annual master table.",
        )

        save_csv(
            result,
            out / "annual_metrics_all_long.csv",
        )
        save_csv(
            ref,
            out / "reference_cells_all.csv",
        )
        save_csv(
            ev,
            out / "events_continuous_all.csv",
        )
        save_csv(
            ev.loc[ev.cross_year.eq(True)],
            out / "cross_year_events_all.csv",
        )
        save_csv(
            result.loc[~result.annual_complete],
            out / "annual_rows_requiring_review.csv",
        )

        for scheme in SCHEMES:
            sub = result[result.scheme.eq(scheme)]

            base_cols = [
                c for c in result.columns
                if c not in METRICS + EXTRA + [
                    "heatwave_type",
                    "unknown_classification_days",
                ]
            ]

            wide = sub[
                sub.heatwave_type.eq("day")
            ][base_cols].copy()

            for kind in TYPES:
                cols = METRICS + EXTRA + [
                    "unknown_classification_days"
                ]

                b = sub[
                    sub.heatwave_type.eq(kind)
                ][["city", "cell_id", "year"] + cols]

                b = b.rename(
                    columns={
                        c: f"{c}_{kind}" for c in cols
                    }
                )

                wide = wide.merge(
                    b,
                    on=["city", "cell_id", "year"],
                    validate="one_to_one",
                )

            require(
                len(wide) == EXPECTED_POINTS * len(YEARS),
                "Abnormal number of rows in the wide table.",
            )

            save_csv(
                wide,
                out / f"annual_{scheme}_wide.csv",
            )

        require(
            sha256(MANIFEST) == manifest_hash,
            "Input manifest changed during the run.",
        )

        summary.update(
            status="calculation_complete",
            cities=len(m),
            points=len(ref),
            baseline_eligible_points=int(
                ref.baseline_eligible.sum()
            ),
            annual_rows=len(result),
            confirmed_events=len(ev),
            cross_year_events=int(
                ev.cross_year.eq(True).sum()
            ),
            annual_rows_requiring_review=int(
                (~result.annual_complete).sum()
            ),
        )

        save()

    except (Exception, KeyboardInterrupt) as exc:
        summary.update(
            status="failed_or_interrupted",
            error=repr(exc),
        )
        save()

        raise

    print("Output folder:", out)

    return result, out


HEATWAVE_ANNUAL_ALL, HEATWAVE_OUTPUT_DIR = run_all()